In [1]:
from pathlib import Path
import re
from collections import Counter

from pypdf import PdfReader
from transformers import pipeline

# 1) PDF -> text extraction 
# -----------------------------
def extract_text_from_pdf(pdf_path: str) -> dict:
    """
    Extracts per-page text using pypdf. Good for text-based PDFs.
    Returns dict with pages_text list and full_text.
    """
    from pypdf import PdfReader

    reader = PdfReader(pdf_path)
    pages_text = []
    for i, page in enumerate(reader.pages):
        txt = page.extract_text() or ""
        # normalize common PDF weirdness a bit
        txt = txt.replace("\u00ad", "")  # soft hyphen
        pages_text.append(txt)

    full_text = "\n\n".join(pages_text).strip()
    return {"pages_text": pages_text, "full_text": full_text, "num_pages": len(pages_text)}


/Users/dimtriospanagoulias/miniconda3/envs/LLMs/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2) sanity checks
def extraction_report(text: str) -> dict:
    words = re.findall(r"\b\w+\b", text)
    printable_ratio = sum(ch.isprintable() for ch in text) / max(1, len(text))
    return {
        "chars": len(text),
        "words": len(words),
        "printable_ratio": round(printable_ratio, 4),
        "preview": text[:800].replace("\n", "\\n")
    }


In [3]:
# 3) Chunking for emotion model (tokenizer-aware)
def chunk_by_tokens(text: str, tokenizer, max_tokens: int, overlap_tokens: int = 80):
    """
    Chunk text by tokens, ensuring no chunk exceeds max_tokens.
    Accounts for special tokens that will be added during inference.
    """
    ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    while start < len(ids):
        end = min(len(ids), start + max_tokens)
        chunk_ids = ids[start:end]
        chunks.append(tokenizer.decode(chunk_ids, skip_special_tokens=True))
        if end == len(ids):
            break
        start = max(0, end - overlap_tokens)
    return chunks


In [4]:
def validate_chunks(chunks,tokenizer, max_length):
    problematic =[]
    for i, chunk in enumerate(chunks):
        tokens = tokenizer.encode(chunk, add_special_tokens=True)
        if len(tokens) > max_length:
            problematic.append((i, len(tokens)))
    return problematic
    

In [5]:
def emotion_from_pdf(
    pdf_path: str,
    model_name: str = "j-hartmann/emotion-english-distilroberta-base",
    device: int = -1,
    overlap_tokens: int = 80,
    out_txt: str = "emotion_results.txt",
):
    extracted = extract_text_from_pdf(pdf_path)
    text = extracted["full_text"]
    print("Extraction sanity:", extraction_report(text))

    clf = pipeline("text-classification", model=model_name, top_k=None, device=device)
    tok = clf.tokenizer

    max_in = getattr(tok, "model_max_length", 512)
    if not isinstance(max_in, int) or max_in > 100000:
        max_in = 512

    # Reserve more space for special tokens (CLS, SEP, etc.)
    max_tokens = max(64, max_in - 32)  # Increased margin from 16 to 32
    chunks = chunk_by_tokens(text, tok, max_tokens=max_tokens, overlap_tokens=overlap_tokens)
    print(f"Chunks: {len(chunks)} (max_tokens={max_tokens}, overlap={overlap_tokens})")

    # Validate chunks
    problematic = validate_chunks(chunks, tok, max_in)
    if problematic:
        print(f"WARNING: {len(problematic)} chunks exceed max length!")
        for idx, length in problematic[:5]:  # Show first 5
            print(f"  Chunk {idx}: {length} tokens (max: {max_in})")
        print("  Truncation will occur during inference.")

    results = []
    for i, ch in enumerate(chunks, 1):
        # Explicitly truncate to be safe
        preds = clf(ch, truncation=True, max_length=max_in)[0]
        top = max(preds, key=lambda d: d["score"])
        results.append({"chunk": i, "top_emotion": top["label"], "score": float(top["score"])})

    counts = Counter(r["top_emotion"] for r in results)

    # Save
    out_path = Path(out_txt)
    lines = [
        f"MODEL: {model_name}\n",
        f"PDF: {pdf_path}\n",
        f"Chunks: {len(chunks)} (max_tokens={max_tokens}, overlap={overlap_tokens})\n\n",
        "Per-chunk top emotion:\n",
    ]
    for r in results:
        lines.append(f"Chunk {r['chunk']:>3}: {r['top_emotion']:<15} score={r['score']:.4f}\n")

    lines.append("\nCounts:\n")
    for lbl, cnt in counts.most_common():
        lines.append(f"{lbl}: {cnt}\n")

    out_path.write_text("".join(lines), encoding="utf-8")
    print("Saved:", out_path)

    return results, counts

In [ ]:
<!-- # --- Run  -->
pdf_path = "eswa.pdf"
emotion_from_pdf(pdf_path)

Extraction sanity: {'chars': 7133, 'words': 1182, 'printable_ratio': 0.983, 'preview': 'Expert Systems With Applications 248 (2024) 123375\\n11D.P. Panagoulias et al.\\nTable 5\\nClusters of perceived usefulness — AI literacy.\\nQuestion 𝜔1(𝜀𝜗𝜛𝜚 ) 𝜔2(𝜀𝜗𝜛𝜚 ) Q(Average)\\nQ1 3.40 3.0 3.20\\nQ2 3.33 3.25 3.29\\nQ3 3.96 2.91 3.70\\nQ4 4.11 3.25 3.435\\nQ5 4.55 2.75 3.65\\nTotalCount 27 12\\nAverage 3.87 3.03\\nTable 6\\nPerceived usefulness and associated AI literacy, survey questions.\\nQuestion Perceived usefulness\\nQ1 Can you indicate your level of knowledge on diagnostic\\nmedicine\\nQ2 Computer vision is a field of artificial intelligence that trains\\ncomputers to interpret and understand the visual world.\\nWould you trust it as a feature in driving automation\\nQ3 AI can add value by either automating, assisting or\\naugmenting the work of clinicians and staff. Many repetitive\\ntasks will become fully automated, '}


/Users/dimtriospanagoulias/miniconda3/envs/LLMs/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Token indices sequence length is longer than the specified maximum sequence length for this model (1698 > 512). Running this sequence through the model will result in indexing errors


Chunks: 5 (max_tokens=480, overlap=80)
Saved: emotion_results.txt


([{'chunk': 1, 'top_emotion': 'neutral', 'score': 0.8930954337120056},
  {'chunk': 2, 'top_emotion': 'neutral', 'score': 0.9491187930107117},
  {'chunk': 3, 'top_emotion': 'neutral', 'score': 0.9135758876800537},
  {'chunk': 4, 'top_emotion': 'neutral', 'score': 0.8448018431663513},
  {'chunk': 5, 'top_emotion': 'neutral', 'score': 0.9290348887443542}],
 Counter({'neutral': 5}))